In [ ]:
%pip install genanki requests

from pathlib import Path
from collections import Counter
from difflib import SequenceMatcher
from urllib.parse import urljoin

import html
import math
import re
import unicodedata

import genanki
import pandas as pd
import requests

# -----------------------
# Paths
# -----------------------

VOCABULARY_PATH = Path("./vocabulary_frozen.pkl")
OCCURRENCES_PATH = Path("./vocabulary_occurrences_frozen.pkl")
MANUAL_REVIEW_PATH = Path("./manual_lexeme_review.csv")
QURANMORPH_PATH = Path("./quran-morph.csv")
QURAN_TEXT_PATH = Path("./quran-uthmani.txt")
QF_PATH = Path("./QF_Database.csv")

FINAL_LEXICON_PATH = Path("./final_anki_lexicon.csv")

MEDIA_DIR = Path("./anki_media_v3")
MEDIA_DIR.mkdir(exist_ok=True)

TEST_APKG_PATH = Path("./quran_vocabulary_test.apkg")
FULL_APKG_PATH = Path("./complete_quran_vocabulary.apkg")


# Quran Foundation word-by-word audio CDN
AUDIO_BASE_URL = "https://audio.qurancdn.com/"


# Stable Anki IDs.
# Do not change these after you begin importing the deck.
DECK_ID = 2059400110
MODEL_ID = 1607392319

DECK_NAME = "Complete Qur'an Vocabulary"

In [ ]:
vocabulary = pd.read_pickle(
    VOCABULARY_PATH
)

vocabulary_occurrences = pd.read_pickle(
    OCCURRENCES_PATH
)

manual_review = pd.read_csv(
    MANUAL_REVIEW_PATH,
    keep_default_na=False,
    encoding="utf-8-sig",
)


# Ensure Vocabulary ID is the index everywhere.
if "Vocabulary ID" in vocabulary.columns:
    vocabulary = vocabulary.set_index(
        "Vocabulary ID"
    )

vocabulary.index.name = "Vocabulary ID"


manual_review = manual_review.set_index(
    "Vocabulary ID"
)


# ------------------------------------------------------------
# Recover the COMPLETE orthographic Qur'anic word.
#
# vocabulary_occurrences["Arabic"] is a QAC lexical segment.
# It can therefore be only part of a written word:
#
#     Arabic        = فِي
#     Quranic Word  = فِيمَا
#
# The audio belongs to the complete orthographic word, not to
# the individual QAC segment.
# ------------------------------------------------------------

key = [
    "Surah",
    "Ayah",
    "Word",
]


quranmorph_words = pd.read_csv(
    QURANMORPH_PATH,
    usecols=[
        "surah_number",
        "verse_number",
        "word_position",
        "word",
    ],
    keep_default_na=False,
).rename(
    columns={
        "surah_number": "Surah",
        "verse_number": "Ayah",
        "word_position": "Word",
        "word": "Quranic Word",
    }
)


for column in key:
    quranmorph_words[column] = pd.to_numeric(
        quranmorph_words[column],
        errors="raise",
    ).astype(int)


assert not quranmorph_words.duplicated(
    key
).any()


vocabulary_occurrences = (
    vocabulary_occurrences
    .drop(
        columns=[
            "Quranic Word",
            "Audio",
        ],
        errors="ignore",
    )
    .merge(
        quranmorph_words,
        on=key,
        how="left",
        validate="many_to_one",
    )
)


missing_quranic_words = (
    vocabulary_occurrences[
        "Quranic Word"
    ]
    .astype(str)
    .str.strip()
    .eq("")
    .sum()
)

assert missing_quranic_words == 0, (
    "Some vocabulary occurrences could not be matched "
    "to a complete Qur'anic word."
)


# ------------------------------------------------------------
# Word-by-word audio
#
# Do NOT use the old Audio column from QF_Database.csv here.
# In the current project export, those paths are not aligned
# to the QAC Word coordinate.
#
# Construct the canonical word-audio filename directly from
# (Surah, Ayah, Word).
# ------------------------------------------------------------

def make_word_audio_path(row):
    return (
        f"wbw/"
        f"{int(row['Surah']):03d}_"
        f"{int(row['Ayah']):03d}_"
        f"{int(row['Word']):03d}.mp3"
    )


vocabulary_occurrences[
    "Audio"
] = (
    vocabulary_occurrences.apply(
        make_word_audio_path,
        axis=1,
    )
)


print(
    "Vocabulary lexemes:",
    len(vocabulary)
)

print(
    "Occurrences:",
    len(vocabulary_occurrences)
)

print(
    "Manual-review rows:",
    len(manual_review)
)

print(
    "Missing complete Qur'anic words:",
    missing_quranic_words,
)


assert vocabulary.index.is_unique
assert manual_review.index.is_unique

assert (
    vocabulary_occurrences[
        "Vocabulary ID"
    ].nunique()
    == len(vocabulary)
)


In [ ]:
vocabulary_occurrences[
    [
        "Surah",
        "Ayah",
        "Word",
        "Segment",
        "Arabic",
        "Quranic Word",
        "Audio",
    ]
].head(10)


In [ ]:
def is_blank(value):
    if value is None:
        return True

    try:
        if pd.isna(value):
            return True
    except (TypeError, ValueError):
        pass

    return (
        isinstance(value, str)
        and value.strip() == ""
    )


def as_bool(value):
    if isinstance(value, bool):
        return value

    if is_blank(value):
        return False

    return (
        str(value)
        .strip()
        .lower()
        in {
            "true",
            "1",
            "yes",
            "y",
        }
    )


def ensure_list(value):
    if isinstance(value, list):
        return value

    if isinstance(value, tuple):
        return list(value)

    if isinstance(value, set):
        return list(value)

    if is_blank(value):
        return []

    return [value]


def contains_arabic(text):
    if not isinstance(text, str):
        return False

    return any(
        "\u0600" <= char <= "\u06ff"
        for char in text
    )


def display_list(value, separator="، "):
    values = ensure_list(value)

    return separator.join(
        str(item)
        for item in values
        if not is_blank(item)
    )


def display_scalar_or_list(value):
    if isinstance(
        value,
        (list, tuple, set)
    ):
        return display_list(value)

    if is_blank(value):
        return ""

    return str(value)

In [ ]:
# ============================================================
# Tanzil waqf-segment context
# ============================================================
#
# Source:
#   ./quran-simple.txt
#
# Downloaded from Tanzil with "Include pause marks" enabled.
#
# We use Tanzil only to determine pause-bounded context spans
# and to display the corresponding simple-script Arabic.
#
# Waqf signs treated as usable boundaries:
#   ۘ  compulsory pause
#   ۚ  permissible pause
#   ۗ  stopping preferred
#   ۖ  continuation preferred, but stopping permissible
#
# We do NOT split at:
#   ۙ  impermissible pause
#   ۛ  interchangeable paired pauses
#
# Ayah endings are always boundaries.
# ============================================================


WAQF_CONTINUE_PREFERRED = "\u06D6"  # ۖ
WAQF_STOP_PREFERRED = "\u06D7"      # ۗ
WAQF_REQUIRED = "\u06D8"            # ۘ
WAQF_NO_STOP = "\u06D9"             # ۙ
WAQF_PERMISSIBLE = "\u06DA"         # ۚ
WAQF_INTERCHANGEABLE = "\u06DB"     # ۛ

VALID_WAQF_BOUNDARIES = {
    WAQF_CONTINUE_PREFERRED,
    WAQF_STOP_PREFERRED,
    WAQF_REQUIRED,
    WAQF_PERMISSIBLE,
}

ALL_WAQF_SIGNS = {
    WAQF_CONTINUE_PREFERRED,
    WAQF_STOP_PREFERRED,
    WAQF_REQUIRED,
    WAQF_NO_STOP,
    WAQF_PERMISSIBLE,
    WAQF_INTERCHANGEABLE,
}

# These may be present if they were included in the Tanzil download.
# They are annotations, not lexical words.
OTHER_QURAN_SIGNS = {
    "\u06DC",  # ۜ small high seen / sakt annotation
    "\u06DE",  # ۞ rub el hizb
    "\u06E9",  # ۩ sajdah
}

# ------------------------------------------------------------
# Rare Uthmani/QuranMorph word-boundary differences
# ------------------------------------------------------------
#
# These are the four genuine word-boundary mismatches remaining
# after ignoring the standalone sakt sign ۜ.
#
# Keys are explicit verse coordinates so this remains auditable
# rather than applying a broad linguistic rewrite everywhere.
#
# Each tuple says:
#     merge these adjacent Tanzil display words into one
#     QuranMorph/QF word coordinate.
#
# The visible Arabic keeps the internal space.
# ------------------------------------------------------------

TANZIL_WORD_MERGES = {
    (2, 181): [
        ("بَعْدَ", "مَا"),
    ],
    (8, 6): [
        ("بَعْدَ", "مَا"),
    ],
    (13, 37): [
        ("بَعْدَ", "مَا"),
    ],
    (37, 130): [
        ("إِلْ", "يَاسِينَ"),
    ],
}


def boundary_compare_key(text):
    """
    Arabic comparison key used only to locate the explicit
    word-boundary overrides above.

    Removes vocalization/Qur'anic marks while preserving letters.
    """
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize(
        "NFD",
        text,
    )

    text = (
        text
        .replace("ٱ", "ا")
        .replace("ى", "ي")
        .replace("ـ", "")
    )

    return "".join(
        char
        for char in text
        if (
            unicodedata.category(char) != "Mn"
            and char not in ALL_WAQF_SIGNS
            and char not in OTHER_QURAN_SIGNS
        )
    )


def apply_tanzil_word_merges(
    surah,
    ayah,
    words,
    boundary_after,
):
    """
    Apply the small set of audited Tanzil/QuranMorph word-boundary
    reconciliations while preserving the learner-facing Tanzil text.

    Also remaps pause-boundary positions after a merge.
    """
    rules = TANZIL_WORD_MERGES.get(
        (surah, ayah),
        [],
    )

    if not rules:
        return words, boundary_after

    current_words = list(words)
    current_boundaries = dict(boundary_after)

    for rule in rules:
        rule_keys = [
            boundary_compare_key(part)
            for part in rule
        ]

        match_start = None

        for start in range(
            0,
            len(current_words) - len(rule) + 1,
        ):
            candidate_keys = [
                boundary_compare_key(part)
                for part in current_words[
                    start:
                    start + len(rule)
                ]
            ]

            if candidate_keys == rule_keys:
                match_start = start
                break

        if match_start is None:
            raise RuntimeError(
                "Expected Tanzil word-boundary override was not found "
                f"at {surah}:{ayah}: {' | '.join(rule)}"
            )

        old_start_pos = match_start + 1
        old_end_pos = match_start + len(rule)

        # A valid pause boundary inside a word that QuranMorph treats
        # as one coordinate would be unsafe to collapse.
        internal_boundaries = [
            position
            for position in current_boundaries
            if old_start_pos <= position < old_end_pos
        ]

        if internal_boundaries:
            raise RuntimeError(
                "Cannot merge Tanzil words across a waqf boundary at "
                f"{surah}:{ayah}; positions {internal_boundaries}"
            )

        merged_display = " ".join(
            current_words[
                match_start:
                match_start + len(rule)
            ]
        )

        new_words = (
            current_words[:match_start]
            + [merged_display]
            + current_words[
                match_start + len(rule):
            ]
        )

        # Remap old 1-based word positions onto the new list.
        shift = len(rule) - 1
        new_boundaries = {}

        for old_position, mark in current_boundaries.items():
            if old_position < old_start_pos:
                new_position = old_position

            elif old_position <= old_end_pos:
                # A boundary after the final member becomes a boundary
                # after the newly merged display word.
                new_position = old_start_pos

            else:
                new_position = old_position - shift

            new_boundaries[
                new_position
            ] = mark

        current_words = new_words
        current_boundaries = new_boundaries

    return (
        current_words,
        current_boundaries,
    )



def load_tanzil_ayahs(path):
    """
    Return one row per ayah with columns:
        Surah, Ayah, Tanzil Text

    Supports both:
      1) plain Tanzil text: one ayah per line
      2) surah|ayah|text
    """
    raw_lines = path.read_text(
        encoding="utf-8"
    ).splitlines()

    lines = [
        line.strip()
        for line in raw_lines
        if (
            line.strip()
            and not line.lstrip().startswith("#")
        )
    ]

    pipe_pattern = re.compile(
        r"^(\d+)\|(\d+)\|(.*)$"
    )

    pipe_matches = [
        pipe_pattern.match(line)
        for line in lines
    ]

    if (
        lines
        and all(
            match is not None
            for match in pipe_matches
        )
    ):
        rows = [
            {
                "Surah": int(match.group(1)),
                "Ayah": int(match.group(2)),
                "Tanzil Text": match.group(3).strip(),
            }
            for match in pipe_matches
        ]

        result = pd.DataFrame(rows)

    else:
        ayah_keys = (
            quranmorph_words[
                ["Surah", "Ayah"]
            ]
            .drop_duplicates()
            .sort_values(
                ["Surah", "Ayah"]
            )
            .reset_index(drop=True)
        )

        if len(lines) != len(ayah_keys):
            raise ValueError(
                "Could not align quran-simple.txt to the Qur'an. "
                f"Found {len(lines)} text lines but expected "
                f"{len(ayah_keys)} ayahs. "
                "Use Tanzil plain text or surah|ayah|text format."
            )

        result = ayah_keys.copy()
        result["Tanzil Text"] = lines

    assert not result.duplicated(
        ["Surah", "Ayah"]
    ).any()

    return result


def split_tanzil_ayah(
    surah,
    ayah,
    text,
):
    """
    Parse one Tanzil ayah into lexical words and usable
    pause boundaries.

    Returns:
        words:
            list of lexical word strings

        boundary_after:
            {word_position: pause_mark}

    A boundary at the end of the ayah is always added.
    """
    tokens = text.split()

    # Tanzil's standard quran-simple file includes the basmala
    # at the beginning of the first ayah of most surahs.
    # QuranMorph/QF do not count those four words as part of
    # the ayah, so mirror Tanzil's own sample-code convention.
    if (
        ayah == 1
        and surah not in {1, 9}
    ):
        lexical_seen = 0
        cut_index = 0

        for token_index, token in enumerate(tokens):
            cleaned = "".join(
                char
                for char in token
                if (
                    char not in ALL_WAQF_SIGNS
                    and char not in OTHER_QURAN_SIGNS
                )
            )

            if cleaned:
                lexical_seen += 1

            if lexical_seen == 4:
                cut_index = token_index + 1
                break

        if lexical_seen == 4:
            tokens = tokens[cut_index:]

    words = []
    boundary_after = {}
    last_word_position = 0

    for token in tokens:
        marks = [
            char
            for char in token
            if char in ALL_WAQF_SIGNS
        ]

        cleaned = "".join(
            char
            for char in token
            if (
                char not in ALL_WAQF_SIGNS
                and char not in OTHER_QURAN_SIGNS
            )
        )

        if cleaned:
            words.append(cleaned)
            last_word_position = len(words)

        # A pause sign normally appears as a standalone token
        # after the preceding word, but this also handles an
        # attached sign if present.
        for mark in marks:
            if (
                mark in VALID_WAQF_BOUNDARIES
                and last_word_position > 0
            ):
                boundary_after[
                    last_word_position
                ] = mark

    if words:
        # Ayah end is always a valid context boundary.
        boundary_after.setdefault(
            len(words),
            "",
        )

    words, boundary_after = (
        apply_tanzil_word_merges(
            surah,
            ayah,
            words,
            boundary_after,
        )
    )

    return words, boundary_after


tanzil_ayahs = load_tanzil_ayahs(
    QURAN_TEXT_PATH
)


pause_mark_count = sum(
    sum(
        text.count(mark)
        for mark in ALL_WAQF_SIGNS
    )
    for text in tanzil_ayahs[
        "Tanzil Text"
    ]
)

if pause_mark_count == 0:
    raise ValueError(
        "No Tanzil pause marks were detected in quran-simple.txt. "
        "Please redownload it with 'Include pause marks' enabled."
    )


# ------------------------------------------------------------
# QF word-by-word English
# ------------------------------------------------------------

qf_database = pd.read_csv(
    QF_PATH,
    keep_default_na=False,
)

qf_glosses = (
    qf_database[
        key + ["English"]
    ]
    .drop_duplicates(
        key
    )
    .copy()
)

assert not qf_glosses.duplicated(
    key
).any()

qf_gloss_lookup = {
    (
        int(row.Surah),
        int(row.Ayah),
        int(row.Word),
    ): str(row.English).strip()

    for row in qf_glosses.itertuples(
        index=False
    )
}


# ------------------------------------------------------------
# Build one context row per orthographic Qur'anic word.
# ------------------------------------------------------------

expected_word_counts = (
    quranmorph_words
    .groupby(
        ["Surah", "Ayah"]
    )
    .size()
    .rename("Expected Word Count")
)

context_rows = []
count_mismatches = []


for ayah_row in tanzil_ayahs.itertuples(
    index=False
):
    surah = int(
        getattr(ayah_row, "Surah")
    )

    ayah = int(
        getattr(ayah_row, "Ayah")
    )

    text = getattr(
        ayah_row,
        "_2",  # third tuple field: Tanzil Text
    )

    words, boundary_after = (
        split_tanzil_ayah(
            surah,
            ayah,
            text,
        )
    )

    expected_count = int(
        expected_word_counts.loc[
            (surah, ayah)
        ]
    )

    if len(words) != expected_count:
        count_mismatches.append(
            {
                "Surah": surah,
                "Ayah": ayah,
                "Expected": expected_count,
                "Tanzil": len(words),
                "Text": text,
            }
        )
        continue

    segment_start = 1

    for segment_end in sorted(
        boundary_after
    ):
        if segment_end < segment_start:
            continue

        segment_words = words[
            segment_start - 1:
            segment_end
        ]

        ending_mark = (
            boundary_after[
                segment_end
            ]
        )

        segment_glosses = [
            qf_gloss_lookup.get(
                (
                    surah,
                    ayah,
                    word_position,
                ),
                "",
            )
            for word_position
            in range(
                segment_start,
                segment_end + 1,
            )
        ]

        context_english = " / ".join(
            gloss
            for gloss in segment_glosses
            if gloss
        )

        for word_position in range(
            segment_start,
            segment_end + 1,
        ):
            local_index = (
                word_position
                - segment_start
            )

            before = " ".join(
                segment_words[
                    :local_index
                ]
            )

            target = segment_words[
                local_index
            ]

            after_words = list(
                segment_words[
                    local_index + 1:
                ]
            )

            after = " ".join(
                after_words
            )

            # Preserve the pause sign visually at the end
            # of the context segment.
            if ending_mark:
                if after:
                    after = (
                        after
                        + " "
                        + ending_mark
                    )
                else:
                    after = ending_mark

            context_rows.append(
                {
                    "Surah": surah,
                    "Ayah": ayah,
                    "Word": word_position,

                    "Waqf Start Word":
                        segment_start,

                    "Waqf End Word":
                        segment_end,

                    "Waqf Word Count":
                        (
                            segment_end
                            - segment_start
                            + 1
                        ),

                    "Context Before Arabic":
                        before,

                    "Context Target Arabic":
                        target,

                    "Context After Arabic":
                        after,

                    "Context English":
                        context_english,

                    "Waqf Ending Mark":
                        ending_mark,
                }
            )

        segment_start = (
            segment_end + 1
        )


if count_mismatches:
    mismatch_df = pd.DataFrame(
        count_mismatches
    )

    display(
        mismatch_df.head(20)
    )

    raise RuntimeError(
        "Unexpected Tanzil/QuranMorph word-count alignment "
        f"failure for {len(count_mismatches)} ayahs after applying "
        "the audited Uthmani boundary fixes. "
        "The first mismatches are displayed above."
    )


waqf_context_map = pd.DataFrame(
    context_rows
)


assert not waqf_context_map.duplicated(
    key
).any()

assert len(
    waqf_context_map
) == len(
    quranmorph_words
)


# Join context metadata onto every QAC vocabulary segment.
context_columns = [
    "Waqf Start Word",
    "Waqf End Word",
    "Waqf Word Count",
    "Context Before Arabic",
    "Context Target Arabic",
    "Context After Arabic",
    "Context English",
    "Waqf Ending Mark",
]


vocabulary_occurrences = (
    vocabulary_occurrences
    .drop(
        columns=context_columns,
        errors="ignore",
    )
    .merge(
        waqf_context_map[
            key + context_columns
        ],
        on=key,
        how="left",
        validate="many_to_one",
    )
)


assert (
    vocabulary_occurrences[
        "Waqf Word Count"
    ]
    .notna()
    .all()
)


print(
    "Tanzil ayahs:",
    len(tanzil_ayahs)
)

print(
    "Pause marks detected:",
    pause_mark_count
)

print(
    "Waqf-context word rows:",
    len(waqf_context_map)
)

print(
    "Waqf segment length range:",
    int(
        waqf_context_map[
            "Waqf Word Count"
        ].min()
    ),
    "to",
    int(
        waqf_context_map[
            "Waqf Word Count"
        ].max()
    ),
    "words",
)


display(
    waqf_context_map.head(10)
)


In [ ]:
manual_columns = [
    "Manual Headword",
    "Manual Plural",
    "Manual Definition",
    "Manual Notes",
    "Reviewed",
]

missing_manual_columns = (
    set(manual_columns)
    - set(manual_review.columns)
)

assert not missing_manual_columns, (
    "Missing manual-review columns: "
    f"{missing_manual_columns}"
)


work = vocabulary.copy()

work = work.join(
    manual_review[
        manual_columns
    ],
    how="left",
)


work["Reviewed"] = (
    work["Reviewed"]
    .apply(as_bool)
)

print(
    work["Reviewed"]
    .value_counts()
)

print(
    "\nReviewed manual lexemes:",
    work["Reviewed"].sum()
)

In [ ]:
def canonical_arabic_form(entry):
    """
    Return Wiktionary's canonical vocalized Arabic form
    when available, otherwise its page word.
    """
    if not isinstance(entry, dict):
        return None

    for form_data in entry.get(
        "forms",
        []
    ):
        form = form_data.get("form")

        tags = set(
            form_data.get(
                "tags",
                []
            )
        )

        if (
            "canonical" in tags
            and contains_arabic(form)
        ):
            return form

    word = entry.get("word")

    if contains_arabic(word):
        return word

    return None


def unique_candidate_headword(candidates):
    """
    If every POS-compatible Wiktionary candidate
    resolves to one canonical Arabic headword,
    return it.
    """
    if not isinstance(
        candidates,
        list
    ):
        return None

    headwords = set()

    for candidate in candidates:
        if not isinstance(
            candidate,
            dict
        ):
            continue

        entry = candidate.get(
            "entry",
            candidate
        )

        headword = (
            canonical_arabic_form(
                entry
            )
        )

        if headword:
            headwords.add(
                headword
            )

    if len(headwords) == 1:
        return next(
            iter(headwords)
        )

    return None


work[
    "Wiktionary Headword Candidate"
] = (
    work[
        "Wiktionary POS Candidates"
    ]
    .apply(
        unique_candidate_headword
    )
)

In [ ]:
def choose_headword(row):
    # 1. Manual decision
    if (
        row["Reviewed"]
        and not is_blank(
            row["Manual Headword"]
        )
    ):
        return (
            row["Manual Headword"].strip(),
            "Manual",
        )

    # 2. Wiktionary
    wiktionary_headword = row[
        "Wiktionary Headword Candidate"
    ]

    if not is_blank(
        wiktionary_headword
    ):
        return (
            wiktionary_headword,
            "Wiktionary",
        )

    # 3. QuranMorph
    quranmorph_forms = ensure_list(
        row.get(
            "QuranMorphMatchForms",
            []
        )
    )

    if len(quranmorph_forms) == 1:
        return (
            quranmorph_forms[0],
            "QuranMorph",
        )

    # 4. QAC
    qac_lemma = row.get(
        "QAC Lemma",
        row.get("Lemma")
    )

    return (
        qac_lemma,
        "QAC",
    )


headword_results = work.apply(
    choose_headword,
    axis=1,
    result_type="expand",
)

headword_results.columns = [
    "Headword",
    "Headword Source",
]

work[
    [
        "Headword",
        "Headword Source",
    ]
] = headword_results

print(
    work[
        "Headword Source"
    ].value_counts()
)

assert not work[
    "Headword"
].apply(is_blank).any()

In [ ]:
def extract_wiktionary_glosses(
    candidates
):
    """
    Extract dictionary glosses from the
    POS-compatible Kaikki/Wiktionary entries.

    Inflection-only / form-of senses are skipped
    where identifiable.
    """
    results = []

    if not isinstance(
        candidates,
        list
    ):
        return results

    for candidate in candidates:
        if not isinstance(
            candidate,
            dict
        ):
            continue

        entry = candidate.get(
            "entry",
            candidate
        )

        if not isinstance(
            entry,
            dict
        ):
            continue

        for sense in entry.get(
            "senses",
            []
        ):
            if not isinstance(
                sense,
                dict
            ):
                continue

            # Avoid a pure inflection/form-of definition
            # when possible.
            if sense.get("form_of"):
                continue

            tags = set(
                sense.get(
                    "tags",
                    []
                )
            )

            if (
                "form-of" in tags
                or "inflection-template"
                in tags
            ):
                continue

            for gloss in sense.get(
                "glosses",
                []
            ):
                if (
                    isinstance(gloss, str)
                    and gloss.strip()
                ):
                    results.append(
                        gloss.strip()
                    )

    # Deduplicate while preserving order.
    return list(
        dict.fromkeys(results)
    )


work[
    "Wiktionary Definition Candidates"
] = (
    work[
        "Wiktionary POS Candidates"
    ]
    .apply(
        extract_wiktionary_glosses
    )
)

In [ ]:
ENGLISH_STOPWORDS = {
    "a",
    "an",
    "the",
    "to",
    "of",
    "and",
    "or",
    "be",
    "is",
    "are",
    "was",
    "were",
    "one",
    "someone",
    "something",
}

CONTENT_POS = {
    "V",
    "N",
    "ADJ",
    "PN",
}


def is_function_lexeme(row):
    """
    Treat particles, conjunctions, pronouns, demonstratives,
    relatives, certainty particles, negatives, etc. as
    grammatical/function lexemes.

    For these, Qur'anic gloss evidence is safer than an
    unrestricted Wiktionary lookup.
    """
    pos = set(
        ensure_list(
            row.get("POS")
        )
    )

    if not pos:
        return False

    return not bool(
        pos & CONTENT_POS
    )


def top_qf_glosses(
    evidence,
    limit=3,
):
    """
    Return a small set of common Qur'anic glosses rather than
    pretending there must be exactly one English translation.
    """
    items = evidence_items(
        evidence
    )

    if not items:
        return []

    try:
        items = sorted(
            items,
            key=lambda item: item[1],
            reverse=True,
        )
    except TypeError:
        pass

    results = []
    seen = set()

    for gloss, count in items:
        if not isinstance(gloss, str):
            continue

        gloss = gloss.strip()

        if not gloss:
            continue

        key = normalize_english(
            gloss
        )

        if key in seen:
            continue

        seen.add(key)
        results.append(gloss)

        if len(results) >= limit:
            break

    return results


def normalize_english(text):
    if not isinstance(
        text,
        str
    ):
        return ""

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s'-]",
        " ",
        text,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    ).strip()

    return text


def english_tokens(text):
    return {
        token
        for token in
        normalize_english(
            text
        ).split()
        if token not in
        ENGLISH_STOPWORDS
    }


def evidence_items(value):
    """
    Return (gloss, frequency) pairs from Definition Evidence.
    """
    if isinstance(
        value,
        Counter
    ):
        return list(
            value.items()
        )

    if isinstance(
        value,
        dict
    ):
        return list(
            value.items()
        )

    if isinstance(
        value,
        str
    ) and value.strip():
        return [
            (value, 1)
        ]

    return []


def fuzzy_token_match(
    token_a,
    token_b,
):
    if token_a == token_b:
        return True

    return (
        SequenceMatcher(
            None,
            token_a,
            token_b,
        ).ratio()
        >= 0.78
    )


def definition_support(
    dictionary_gloss,
    quran_gloss,
):
    """
    Measure whether a dictionary gloss is actually supported by
    the Qur'anic gloss.

    Precision matters heavily here:
        'day' vs 'day'             -> strong
        'Judgment Day' vs 'day'    -> penalized
    """
    dictionary_tokens = list(
        english_tokens(
            dictionary_gloss
        )
    )

    quran_tokens = list(
        english_tokens(
            quran_gloss
        )
    )

    if (
        not dictionary_tokens
        or not quran_tokens
    ):
        return {
            "score": 0.0,
            "precision": 0.0,
        }

    matched_dictionary = 0
    matched_quran = 0

    for dictionary_token in dictionary_tokens:
        if any(
            fuzzy_token_match(
                dictionary_token,
                quran_token,
            )
            for quran_token
            in quran_tokens
        ):
            matched_dictionary += 1

    for quran_token in quran_tokens:
        if any(
            fuzzy_token_match(
                quran_token,
                dictionary_token,
            )
            for dictionary_token
            in dictionary_tokens
        ):
            matched_quran += 1

    precision = (
        matched_dictionary
        / len(dictionary_tokens)
    )

    recall = (
        matched_quran
        / len(quran_tokens)
    )

    sequence = SequenceMatcher(
        None,
        normalize_english(
            dictionary_gloss
        ),
        normalize_english(
            quran_gloss
        ),
    ).ratio()

    score = (
        0.60 * precision
        + 0.25 * recall
        + 0.15 * sequence
    )

    return {
        "score": score,
        "precision": precision,
    }


def score_definition(
    definition,
    evidence
):
    items = evidence_items(
        evidence
    )

    if not items:
        return None

    best_score = 0.0

    for quran_gloss, count in items:
        score = gloss_similarity(
            definition,
            quran_gloss,
        )

        # Tiny preference for meanings supported
        # by more QF occurrences.
        try:
            frequency_bonus = (
                1
                + 0.03
                * math.log1p(
                    float(count)
                )
            )
        except (
            TypeError,
            ValueError,
        ):
            frequency_bonus = 1

        best_score = max(
            best_score,
            score
            * frequency_bonus,
        )

    return best_score

In [ ]:
def most_common_definition_evidence(
    evidence
):
    items = evidence_items(
        evidence
    )

    if not items:
        return None

    try:
        items = sorted(
            items,
            key=lambda item:
                item[1],
            reverse=True,
        )
    except TypeError:
        pass

    return items[0][0]


def choose_definition(row):
    # --------------------------------------------------
    # 1. Manual review always wins
    # --------------------------------------------------

    if (
        row["Reviewed"]
        and not is_blank(
            row["Manual Definition"]
        )
    ):
        return (
            row["Manual Definition"].strip(),
            "Manual",
            1.0,
        )

    qf_glosses = top_qf_glosses(
        row.get(
            "Definition Evidence"
        ),
        limit=3,
    )

    # --------------------------------------------------
    # 2. Function words:
    #    Qur'anic usage is more trustworthy than our
    #    Wiktionary matching for these.
    # --------------------------------------------------

    if is_function_lexeme(row):
        if qf_glosses:
            return (
                "; ".join(
                    qf_glosses[:2]
                ),
                "Quran Foundation",
                None,
            )

    # --------------------------------------------------
    # 3. Content words:
    #    Accept Wiktionary meanings only when the
    #    Qur'anic evidence actually supports them.
    # --------------------------------------------------

    dictionary_candidates = ensure_list(
        row.get(
            "Wiktionary Definition Candidates"
        )
    )

    supported = []

    for candidate in dictionary_candidates:
        best_support = None

        for quran_gloss in qf_glosses:
            support = definition_support(
                candidate,
                quran_gloss,
            )

            if (
                best_support is None
                or support["score"]
                > best_support["score"]
            ):
                best_support = support

        if best_support is None:
            continue

        # Require most of the dictionary meaning itself
        # to be supported by the Qur'anic gloss.
        if (
            best_support["precision"] >= 0.75
            and best_support["score"] >= 0.45
        ):
            supported.append(
                (
                    candidate,
                    best_support["score"],
                )
            )

    if supported:
        supported.sort(
            key=lambda item: (
                -item[1],
                len(item[0]),
            )
        )

        best_score = supported[0][1]

        # Allow several genuinely supported senses,
        # but don't become exhaustive.
        selected = []

        for definition, score in supported:
            if score < best_score - 0.10:
                continue

            if definition not in selected:
                selected.append(
                    definition
                )

            if len(selected) == 3:
                break

        return (
            "; ".join(selected),
            "Wiktionary + Quran Foundation",
            best_score,
        )

    # --------------------------------------------------
    # 4. Qur'anic fallback
    # --------------------------------------------------

    if qf_glosses:
        return (
            "; ".join(
                qf_glosses[:2]
            ),
            "Quran Foundation",
            None,
        )

    # --------------------------------------------------
    # 5. Last-resort Wiktionary fallback
    # --------------------------------------------------

    if dictionary_candidates:
        return (
            dictionary_candidates[0],
            "Wiktionary fallback",
            None,
        )

    return (
        "",
        "Missing",
        None,
    )


definition_results = work.apply(
    choose_definition,
    axis=1,
    result_type="expand",
)

definition_results.columns = [
    "Definition",
    "Definition Source",
    "Definition Match Score",
]

work[
    [
        "Definition",
        "Definition Source",
        "Definition Match Score",
    ]
] = definition_results

print(
    work[
        "Definition Source"
    ].value_counts()
)

print(
    "\nMissing definitions:",
    work[
        "Definition"
    ].apply(
        is_blank
    ).sum()
)

In [ ]:
FINAL_CASE_DIACRITICS = (
    "\u064b"  # fathatan
    "\u064c"  # dammatan
    "\u064d"  # kasratan
    "\u064e"  # fatha
    "\u064f"  # damma
    "\u0650"  # kasra
)


def strip_final_case_ending(
    text
):
    if not isinstance(
        text,
        str
    ):
        return text

    return re.sub(
        f"[{FINAL_CASE_DIACRITICS}]+$",
        "",
        text,
    )


def extract_plural_candidates(
    candidates
):
    plurals = []

    if not isinstance(
        candidates,
        list
    ):
        return plurals

    for candidate in candidates:
        if not isinstance(
            candidate,
            dict
        ):
            continue

        entry = candidate.get(
            "entry",
            candidate
        )

        if not isinstance(
            entry,
            dict
        ):
            continue

        for form_data in entry.get(
            "forms",
            []
        ):
            if not isinstance(
                form_data,
                dict
            ):
                continue

            form = form_data.get(
                "form"
            )

            tags = set(
                form_data.get(
                    "tags",
                    []
                )
            )

            if (
                "plural" not in tags
                or not contains_arabic(
                    form
                )
            ):
                continue

            # Avoid inflected case/state variants
            # when Wiktionary identifies them.
            if tags & {
                "accusative",
                "genitive",
                "construct",
                "definite",
                "indefinite",
                "oblique",
            }:
                continue

            form = (
                strip_final_case_ending(
                    form
                )
            )

            if form:
                plurals.append(
                    form
                )

    return list(
        dict.fromkeys(
            plurals
        )
    )


work[
    "Wiktionary Plural Candidates"
] = (
    work[
        "Wiktionary POS Candidates"
    ]
    .apply(
        extract_plural_candidates
    )
)

In [ ]:
def choose_plural(row):
    # 1. Manual
    if (
        row["Reviewed"]
        and not is_blank(
            row[
                "Manual Plural"
            ]
        )
    ):
        return (
            row[
                "Manual Plural"
            ].strip(),
            "Manual",
        )

    # Only automatically add plural morphology
    # to lexical categories where it is useful.
    pos = set(
        ensure_list(
            row.get("POS")
        )
    )

    if not (
        pos
        & {
            "N",
            "ADJ",
        }
    ):
        return (
            "",
            "Not applicable",
        )

    candidates = ensure_list(
        row[
            "Wiktionary Plural Candidates"
        ]
    )

    if not candidates:
        return (
            "",
            "None found",
        )

    # Multiple genuine plurals are okay.
    # Keep at most three automatically.
    if len(candidates) <= 3:
        return (
            " / ".join(
                candidates
            ),
            "Wiktionary",
        )

    return (
        "",
        "Ambiguous",
    )


plural_results = work.apply(
    choose_plural,
    axis=1,
    result_type="expand",
)

plural_results.columns = [
    "Plural",
    "Plural Source",
]

work[
    [
        "Plural",
        "Plural Source",
    ]
] = plural_results

print(
    work[
        "Plural Source"
    ].value_counts()
)

In [ ]:
def needs_review(row):
    # Anything manually reviewed is accepted.
    if row["Reviewed"]:
        return False

    if is_blank(
        row["Headword"]
    ):
        return True

    if is_blank(
        row["Definition"]
    ):
        return True

    # Anything that failed normal Wiktionary matching
    # and has not been manually reviewed remains flagged.
    if (
        row.get(
            "Wiktionary Match Status"
        )
        != "matched"
    ):
        return True

    score = row[
        "Definition Match Score"
    ]

    if (
        not is_blank(score)
        and score is not None
    ):
        try:
            if float(score) < 0.15:
                return True
        except (
            TypeError,
            ValueError,
        ):
            pass

    return False


work["Auto QA Flag"] = (
    work.apply(
        needs_review,
        axis=1,
    )
)

print(
    work[
        "Auto QA Flag"
    ].value_counts()
)

In [ ]:
QURANIC_MADDA = "\u0653"
DAGGER_ALIF = "\u0670"

STANDARD_HARAKAT = {
    "\u064b",  # fathatan
    "\u064c",  # dammatan
    "\u064d",  # kasratan
    "\u064e",  # fatha
    "\u064f",  # damma
    "\u0650",  # kasra
    "\u0651",  # shadda
    "\u0652",  # sukun
}


def vocalized_compare_key(text):
    """
    Compare Qur'anic forms while preserving meaningful
    vowel information.

    Contextual final case/mood endings are ignored.
    """
    if not isinstance(text, str):
        return None

    text = text.replace(
        QURANIC_MADDA,
        "",
    )

    text = unicodedata.normalize(
        "NFC",
        text,
    )

    text = (
        text
        .replace("ٱ", "ا")
        .replace(DAGGER_ALIF, "ا")
        .replace("ـ", "")
    )

    # Remove Qur'anic annotation marks while preserving
    # ordinary Arabic harakat.
    text = "".join(
        char
        for char in text
        if (
            unicodedata.category(char)
            != "Mn"
            or char in STANDARD_HARAKAT
        )
    )

    # Initial shadda may result from recitational assimilation
    # rather than lexical morphology.
    if len(text) >= 2 and text[1] == "\u0651":
        text = text[0] + text[2:]

    # Ignore final case/mood vowel.
    text = re.sub(
        r"[\u064b\u064c\u064d\u064e\u064f\u0650\u0652]+$",
        "",
        text,
    )

    return text


def vocalized_similarity(
    a,
    b,
):
    a = vocalized_compare_key(a)
    b = vocalized_compare_key(b)

    if not a or not b:
        return 0.0

    return SequenceMatcher(
        None,
        a,
        b,
    ).ratio()


def arabic_compare_key(text):
    if not isinstance(
        text,
        str
    ):
        return None

    text = text.replace(
        QURANIC_MADDA,
        "",
    )

    text = unicodedata.normalize(
        "NFC",
        text,
    )

    text = (
        text
        .replace("آ", "ءا")
        .replace("أ", "ء")
        .replace("إ", "ء")
        .replace("ؤ", "ء")
        .replace("ئ", "ء")
        .replace("ٱ", "ا")
        .replace(DAGGER_ALIF, "ا")
        .replace("ـ", "")
    )

    text = "".join(
        char
        for char in text
        if unicodedata.category(
            char
        ) != "Mn"
    )

    # Relax final alif maqsurah / ya difference
    text = text.replace(
        "ى",
        "ي",
    )

    return text

In [ ]:
def plural_keys(plural):
    if is_blank(plural):
        return set()

    pieces = re.split(
        r"\s*/\s*|\s*;\s*",
        str(plural),
    )

    return {
        key
        for piece in pieces
        if (
            key :=
            arabic_compare_key(
                piece
            )
        )
    }


def surface_identity_key(text):
    """
    Compare Arabic lexical spelling while ignoring Qur'anic
    orthographic marks and contextual vowel endings.

    This is stricter than fuzzy matching but tolerant of:
        ٱ / ا
        ى / ي
        dagger alif
        harakat
    """
    if not isinstance(text, str):
        return None

    text = text.replace(
        QURANIC_MADDA,
        "",
    )

    text = unicodedata.normalize(
        "NFC",
        text,
    )

    text = (
        text
        .replace("ٱ", "ا")
        .replace("ى", "ي")
        .replace(DAGGER_ALIF, "ا")
        .replace("ـ", "")
    )

    text = "".join(
        char
        for char in text
        if unicodedata.category(char)
        != "Mn"
    )

    return text


def restrict_if_possible(
    group,
    mask,
):
    """
    Apply a preference as a hard filter only if at least
    one candidate satisfies it.
    """
    mask = mask.fillna(False)

    if mask.any():
        return group[mask].copy()

    return group


def rank_occurrences(
    vocabulary_id,
    row,
):
    group = vocabulary_occurrences[
        vocabulary_occurrences[
            "Vocabulary ID"
        ]
        == vocabulary_id
    ].copy()

    if group.empty:
        return group

    # ---------------------------------------------
    # 1. Prefer an orthographic word with no
    #    attached suffix when such an example exists.
    # ---------------------------------------------

    if "SuffixCount" in group.columns:
        group = restrict_if_possible(
            group,
            group["SuffixCount"]
            .fillna(0)
            .eq(0),
        )

    elif (
        "HasPronounSuffix"
        in group.columns
    ):
        group = restrict_if_possible(
            group,
            ~group[
                "HasPronounSuffix"
            ].fillna(False),
        )

    # ---------------------------------------------
    # 2. Prefer an exact LEXICAL SEGMENT match.
    #
    # الذي beats الذين.
    # كان beats inflected alternatives if available.
    # ---------------------------------------------

    headword_key = (
        surface_identity_key(
            row["Headword"]
        )
    )

    group["_lexical_key"] = (
        group["Arabic"]
        .apply(
            surface_identity_key
        )
    )

    group = restrict_if_possible(
        group,
        group["_lexical_key"]
        == headword_key,
    )

    # ---------------------------------------------
    # 3. Prefer the COMPLETE WRITTEN WORD itself to
    #    match the headword.
    #
    # This is the key fix for:
    #
    #     lexical segment = فِي
    #     whole word      = فِيمَا
    #
    # If standalone فِي exists, it wins.
    # ---------------------------------------------

    if "Quranic Word" in group.columns:
        group["_whole_word_key"] = (
            group[
                "Quranic Word"
            ]
            .apply(
                surface_identity_key
            )
        )

        group = restrict_if_possible(
            group,
            group["_whole_word_key"]
            == headword_key,
        )

    # ---------------------------------------------
    # 4. Prefer a word containing only one
    #    vocabulary-bearing QAC segment.
    #
    # This rejects compounds such as:
    #     فِيمَا = فِي + مَا
    #
    # while retaining them as fallbacks when a
    # lexeme never occurs independently.
    # ---------------------------------------------

    if (
        "VocabularySegmentCount"
        in group.columns
    ):
        group = restrict_if_possible(
            group,
            group[
                "VocabularySegmentCount"
            ]
            .fillna(0)
            .eq(1),
        )

    # ---------------------------------------------
    # 5. Prefer words with no prefixes.
    #
    # آمَنَ beats فَآمَنَ if both exist.
    # ---------------------------------------------

    if "PrefixCount" in group.columns:
        group = restrict_if_possible(
            group,
            group["PrefixCount"]
            .fillna(0)
            .eq(0),
        )

    # ---------------------------------------------
    # 6. For nominal lexemes, prefer singular.
    # ---------------------------------------------

    pos = set(
        ensure_list(
            row.get("POS")
        )
    )

    if (
        pos & {"N", "ADJ"}
        and "Number"
        in group.columns
    ):
        group = restrict_if_possible(
            group,
            group["Number"]
            .fillna("")
            .eq("S"),
        )

    # ---------------------------------------------
    # 7. Only now use vocalized similarity.
    # ---------------------------------------------

    group[
        "_vocalized_similarity"
    ] = (
        group["Arabic"]
        .apply(
            lambda form:
                vocalized_similarity(
                    row["Headword"],
                    form,
                )
        )
    )

    # ---------------------------------------------
    # 8. Other quality criteria.
    # ---------------------------------------------

    group["_has_audio_path"] = (
        group["Audio"]
        .fillna("")
        .astype(str)
        .str.strip()
        .ne("")
    )

    if "Clean Occurrence" in group.columns:
        group["_clean"] = (
            group[
                "Clean Occurrence"
            ].fillna(False)
        )
    else:
        group["_clean"] = False

    if (
        "Occurrence Tier"
        in group.columns
    ):
        group["_tier"] = (
            pd.to_numeric(
                group[
                    "Occurrence Tier"
                ],
                errors="coerce",
            )
            .fillna(999)
        )
    else:
        group["_tier"] = 999

    group["_waqf_word_count"] = (
        pd.to_numeric(
            group[
                "Waqf Word Count"
            ],
            errors="coerce",
        )
        .fillna(999)
    )

    group = group.sort_values(
        by=[
            "_vocalized_similarity",
            "_waqf_word_count",
            "_clean",
            "_has_audio_path",
            "_tier",
            "Surah",
            "Ayah",
            "Word",
            "Segment",
        ],
        ascending=[
            False,
            True,
            False,
            False,
            True,
            True,
            True,
            True,
            True,
        ],
    )

    return group


def occurrence_record(
    chosen
):
    cleaned_english = (
        chosen.get(
            "Cleaned English"
        )
    )

    if is_blank(
        cleaned_english
    ):
        cleaned_english = (
            chosen.get(
                "English",
                ""
            )
        )

    audio_path = (
        chosen.get(
            "Audio",
            ""
        )
    )

    if is_blank(audio_path):
        audio_path = ""

    quranic_word = (
        chosen.get(
            "Quranic Word",
            ""
        )
    )

    if is_blank(quranic_word):
        # Defensive fallback only.
        quranic_word = (
            chosen.get(
                "Arabic",
                ""
            )
        )

    return {
        # QAC lexical segment belonging to the lexeme.
        "Lexical Occurrence":
            chosen.get(
                "Arabic",
                ""
            ),

        # Complete orthographic word. This is what the
        # word-level audio corresponds to.
        "Quranic Word":
            quranic_word,

        "Occurrence Gloss":
            cleaned_english,

        "Example Surah":
            int(
                chosen["Surah"]
            ),

        "Example Ayah":
            int(
                chosen["Ayah"]
            ),

        "Example Word":
            int(
                chosen["Word"]
            ),

        "Example Segment":
            int(
                chosen["Segment"]
            ),

        "Example Reference": (
            f"{int(chosen['Surah'])}:"
            f"{int(chosen['Ayah'])}"
        ),

        "Audio Path":
            audio_path,

        "Audio URL": (
            urljoin(
                AUDIO_BASE_URL,
                str(audio_path)
                .lstrip("/"),
            )
            if audio_path
            else ""
        ),

        "Waqf Word Count":
            chosen.get(
                "Waqf Word Count",
                "",
            ),

        "Context Before Arabic":
            chosen.get(
                "Context Before Arabic",
                "",
            ),

        "Context Target Arabic":
            chosen.get(
                "Context Target Arabic",
                quranic_word,
            ),

        "Context After Arabic":
            chosen.get(
                "Context After Arabic",
                "",
            ),

        "Context English":
            chosen.get(
                "Context English",
                "",
            ),
    }


# ------------------------------------------------------------
# Audio availability
#
# A path can be structurally correct but the CDN asset can
# still be missing. Try the next-ranked occurrence if so.
# ------------------------------------------------------------

_audio_availability_cache = {}
_audio_check_session = requests.Session()


def audio_exists(
    audio_path,
):
    if is_blank(audio_path):
        return False

    audio_path = (
        str(audio_path)
        .lstrip("/")
    )

    if (
        audio_path
        in _audio_availability_cache
    ):
        return (
            _audio_availability_cache[
                audio_path
            ]
        )

    url = urljoin(
        AUDIO_BASE_URL,
        audio_path,
    )

    try:
        response = (
            _audio_check_session.get(
                url,
                timeout=15,
                stream=True,
            )
        )

        exists = (
            response.status_code
            == 200
        )

        response.close()

    except requests.RequestException:
        exists = False

    _audio_availability_cache[
        audio_path
    ] = exists

    return exists


def choose_representative_occurrence(
    vocabulary_id,
    row,
):
    """
    Choose the best TEXTUAL occurrence only.

    IMPORTANT:
    Do not make network requests here. The old version checked
    audio availability for every lexeme while building the table,
    which made this cell take many minutes.

    Audio availability / fallback is handled later, only when
    we actually build an Anki package.
    """
    ranked = rank_occurrences(
        vocabulary_id,
        row,
    )

    if ranked.empty:
        return {}

    result = occurrence_record(
        ranked.iloc[0]
    )

    # None = not checked yet.
    result[
        "Audio Available"
    ] = None

    return result


In [ ]:
occurrence_records = []

for vocabulary_id, row in (
    work.iterrows()
):
    result = (
        choose_representative_occurrence(
            vocabulary_id,
            row,
        )
    )

    result[
        "Vocabulary ID"
    ] = vocabulary_id

    occurrence_records.append(
        result
    )


selected_occurrences = (
    pd.DataFrame(
        occurrence_records
    )
    .set_index(
        "Vocabulary ID"
    )
)


# Make rerunning this cell safe in an interactive notebook.
columns_to_replace = (
    selected_occurrences.columns
    .intersection(
        work.columns
    )
)

work = (
    work
    .drop(
        columns=columns_to_replace,
        errors="ignore",
    )
    .join(
        selected_occurrences
    )
)


work[
    [
        "Headword",
        "Plural",
        "Definition",
        "Lexical Occurrence",
        "Quranic Word",
        "Occurrence Gloss",
        "Example Reference",
        "Audio Path",
        "Audio Available",
        "Waqf Word Count",
        "Context Before Arabic",
        "Context Target Arabic",
        "Context After Arabic",
        "Context English",
    ]
].head(25)


In [ ]:
work["POS Display"] = (
    work["POS"]
    .apply(
        lambda value:
            ", ".join(
                str(item)
                for item in
                ensure_list(value)
            )
    )
)


work["Observed Forms Display"] = (
    work["ObservedForms"]
    .apply(
        lambda value:
            "، ".join(
                str(item)
                for item in
                ensure_list(value)
            )
    )
)


work["Root Display"] = (
    work["Root"]
    .apply(
        display_scalar_or_list
    )
)

In [ ]:
# ------------------------------------------------------------
# Qur'an location metadata for Anki tags
# ------------------------------------------------------------

def unique_location_numbers(series):
    values = []

    for value in series:
        if is_blank(value):
            continue

        try:
            number = int(value)
        except (
            TypeError,
            ValueError,
        ):
            continue

        values.append(number)

    return sorted(
        set(values)
    )


required_location_columns = {
    "Hizb",
    "Rub el Hizb",
}

missing_location_columns = (
    required_location_columns
    - set(
        vocabulary_occurrences.columns
    )
)

assert not missing_location_columns, (
    "Missing occurrence-level Qur'an location columns: "
    f"{sorted(missing_location_columns)}"
)


location_metadata = (
    vocabulary_occurrences
    .groupby(
        "Vocabulary ID"
    )
    .agg(
        Hizbs=(
            "Hizb",
            unique_location_numbers,
        ),
        Rubs=(
            "Rub el Hizb",
            unique_location_numbers,
        ),
    )
)


work = (
    work
    .drop(
        columns=[
            "Hizbs",
            "Rubs",
        ],
        errors="ignore",
    )
    .join(
        location_metadata,
        how="left",
    )
)


def make_tags(row):
    tags = [
        "CompleteQuranVocabulary"
    ]

    for surah in ensure_list(
        row.get("Surahs")
    ):
        try:
            tags.append(
                f"Quran::Surah::{int(surah):03d}"
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    for juz in ensure_list(
        row.get("Juzs")
    ):
        try:
            tags.append(
                f"Quran::Juz::{int(juz):02d}"
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    for hizb in ensure_list(
        row.get("Hizbs")
    ):
        try:
            tags.append(
                f"Quran::Hizb::{int(hizb):03d}"
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    for rub in ensure_list(
        row.get("Rubs")
    ):
        try:
            tags.append(
                f"Quran::Rub::{int(rub):03d}"
            )
        except (
            TypeError,
            ValueError,
        ):
            pass

    if row["Reviewed"]:
        tags.append(
            "Source::ManualReview"
        )

    if row["Auto QA Flag"]:
        tags.append(
            "NeedsReview"
        )

    return list(
        dict.fromkeys(tags)
    )


work["Anki Tags"] = (
    work.apply(
        make_tags,
        axis=1,
    )
)


assert (
    work["Hizbs"]
    .apply(
        lambda values:
            all(
                1 <= int(value) <= 60
                for value
                in ensure_list(values)
            )
    )
    .all()
)

assert (
    work["Rubs"]
    .apply(
        lambda values:
            all(
                1 <= int(value) <= 240
                for value
                in ensure_list(values)
            )
    )
    .all()
)


print(
    "Lexemes with Hizb metadata:",
    work["Hizbs"]
    .apply(bool)
    .sum(),
)

print(
    "Lexemes with Rub metadata:",
    work["Rubs"]
    .apply(bool)
    .sum(),
)

display(
    work[
        [
            "Headword",
            "Anki Tags",
        ]
    ].head(10)
)


In [ ]:
final_lexicon = pd.DataFrame(
    index=work.index
)


final_lexicon[
    "Headword"
] = work["Headword"]

final_lexicon[
    "Plural"
] = work["Plural"]

final_lexicon[
    "Definition"
] = work["Definition"]

final_lexicon[
    "Root"
] = work["Root Display"]

final_lexicon[
    "POS"
] = work["POS Display"]

final_lexicon[
    "Frequency"
] = work["Frequency"]

final_lexicon[
    "Frequency Rank"
] = work["Frequency Rank"]

final_lexicon[
    "Quranic Form"
] = work["Quranic Word"]

final_lexicon[
    "Lexical Occurrence"
] = work["Lexical Occurrence"]

final_lexicon[
    "Occurrence Gloss"
] = work["Occurrence Gloss"]

final_lexicon[
    "Waqf Word Count"
] = work["Waqf Word Count"]

final_lexicon[
    "Context Before Arabic"
] = work["Context Before Arabic"]

final_lexicon[
    "Context Target Arabic"
] = work["Context Target Arabic"]

final_lexicon[
    "Context After Arabic"
] = work["Context After Arabic"]

final_lexicon[
    "Context English"
] = work["Context English"]

final_lexicon[
    "Example Reference"
] = work["Example Reference"]

final_lexicon[
    "Example Surah"
] = work["Example Surah"]

final_lexicon[
    "Example Ayah"
] = work["Example Ayah"]

final_lexicon[
    "Example Word"
] = work["Example Word"]

final_lexicon[
    "Example Segment"
] = work["Example Segment"]

final_lexicon[
    "Audio Available"
] = work["Audio Available"]

final_lexicon[
    "Audio Path"
] = work["Audio Path"]

final_lexicon[
    "Audio URL"
] = work["Audio URL"]

final_lexicon[
    "Observed Forms"
] = work[
    "Observed Forms Display"
]

final_lexicon[
    "Headword Source"
] = work[
    "Headword Source"
]

final_lexicon[
    "Definition Source"
] = work[
    "Definition Source"
]

final_lexicon[
    "Definition Match Score"
] = work[
    "Definition Match Score"
]

final_lexicon[
    "Plural Source"
] = work[
    "Plural Source"
]

final_lexicon[
    "Auto QA Flag"
] = work[
    "Auto QA Flag"
]

final_lexicon[
    "Manual Notes"
] = work[
    "Manual Notes"
]

final_lexicon[
    "Tags"
] = work[
    "Anki Tags"
].apply(
    lambda tags:
        " ".join(tags)
)


final_lexicon = (
    final_lexicon
    .sort_values(
        "Frequency Rank"
    )
)

In [ ]:
print(
    "Cards:",
    len(final_lexicon)
)

print(
    "Missing headwords:",
    final_lexicon[
        "Headword"
    ].apply(is_blank).sum()
)

print(
    "Missing definitions:",
    final_lexicon[
        "Definition"
    ].apply(is_blank).sum()
)

print(
    "Cards with plural:",
    (
        ~final_lexicon[
            "Plural"
        ].apply(is_blank)
    ).sum()
)

print(
    "Cards with audio:",
    (
        ~final_lexicon[
            "Audio Path"
        ].apply(is_blank)
    ).sum()
)

print(
    "Still flagged for review:",
    final_lexicon[
        "Auto QA Flag"
    ].sum()
)


assert len(
    final_lexicon
) == len(vocabulary)

assert (
    final_lexicon.index.is_unique
)

assert not (
    final_lexicon[
        "Headword"
    ]
    .apply(is_blank)
    .any()
)

assert (
    final_lexicon[
        "Frequency Rank"
    ].is_unique
)

assert (
    final_lexicon[
        "Frequency Rank"
    ].min()
    == 1
)

assert (
    final_lexicon[
        "Frequency Rank"
    ].max()
    == len(
        final_lexicon
    )
)

In [ ]:
final_lexicon_export = (
    final_lexicon
    .reset_index()
)

final_lexicon_export.to_csv(
    FINAL_LEXICON_PATH,
    index=False,
    encoding="utf-8-sig",
)

print(
    "Saved:",
    FINAL_LEXICON_PATH.resolve()
)

In [ ]:
pd.concat(
    [
        final_lexicon.head(50),
        final_lexicon.sample(
            50,
            random_state=42,
        ),
    ]
)[
    [
        "Headword",
        "Plural",
        "Definition",
        "Root",
        "POS",
        "Frequency",
        "Quranic Form",
        "Occurrence Gloss",
        "Headword Source",
        "Definition Source",
        "Auto QA Flag",
    ]
]

In [ ]:
session = requests.Session()


def download_audio(
    relative_path,
    target_name,
    force=False,
):
    if is_blank(
        relative_path
    ):
        return None

    relative_path = (
        str(relative_path)
        .lstrip("/")
    )

    url = urljoin(
        AUDIO_BASE_URL,
        relative_path,
    )

    target = (
        MEDIA_DIR
        / target_name
    )

    if (
        target.exists()
        and not force
    ):
        return target

    try:
        response = session.get(
            url,
            timeout=30,
        )

        response.raise_for_status()

        target.write_bytes(
            response.content
        )

        return target

    except Exception as exc:
        print(
            f"Audio failed: "
            f"{relative_path}: "
            f"{exc}"
        )

        return None

import hashlib


def audio_media_name(
    vocabulary_id,
    example_reference,
    audio_path,
):
    """
    Deterministic media filename.

    Audio Path itself encodes surah/ayah/word, so we do not need
    to depend on a separate Example Word column.
    """
    identity = (
        f"{vocabulary_id}|"
        f"{example_reference}|"
        f"{audio_path}"
    )

    digest = hashlib.sha1(
        identity.encode(
            "utf-8"
        )
    ).hexdigest()[:12]

    return (
        f"quran_vocab_"
        f"{digest}.mp3"
    )


In [ ]:
MODEL_FIELDS = [
    "VocabularyID",
    "Headword",
    "Plural",
    "Definition",
    "Root",
    "POS",
    "Frequency",
    "FrequencyRank",
    "QuranicForm",
    "OccurrenceGloss",
    "ExampleReference",
    "Audio",
    "ObservedForms",
    "HeadwordSource",
    "DefinitionSource",
    "NeedsReview",
]

anki_model = genanki.Model(
    MODEL_ID,
    "Complete Quran Vocabulary Model",

    fields=[
        {"name": field}
        for field
        in MODEL_FIELDS
    ],

    templates=[
        {
            "name": "Arabic → English",

            "qfmt": """
<div class="card-container">

    <div class="headword arabic">
        {{Headword}}
    </div>

    {{#Plural}}
    <div class="plural arabic">
        {{Plural}}
    </div>
    {{/Plural}}

</div>
""",

            "afmt": """
<div class="card-container">

    <div class="headword arabic">
        {{Headword}}
    </div>

    {{#Plural}}
    <div class="plural arabic">
        {{Plural}}
    </div>
    {{/Plural}}

    <hr id="answer">

    <div class="definition">
        {{Definition}}
    </div>

    <div class="metadata">
        {{#Root}}
        <div>
            <span class="label">Root</span>
            <span class="arabic root">{{Root}}</span>
        </div>
        {{/Root}}

        {{#POS}}
        <div>
            <span class="label">POS</span>
            {{POS}}
        </div>
        {{/POS}}
    </div>

    {{#QuranicForm}}
    <div class="occurrence">

        <div class="occurrence-label">
            Qur'anic context
        </div>

        <div class="occurrence-arabic arabic">
            {{QuranicForm}}
        </div>

        {{Audio}}

        {{#OccurrenceGloss}}
        <div class="occurrence-gloss context-english">
            {{OccurrenceGloss}}
        </div>
        {{/OccurrenceGloss}}

        <div class="reference">
            Qur'an {{ExampleReference}}
        </div>

        <div class="reference">
            Pause segmentation: Tanzil.net
        </div>

    </div>
    {{/QuranicForm}}

    <div class="footer">
        Frequency: {{Frequency}}
        · Rank: {{FrequencyRank}}
    </div>

</div>
""",
        }
    ],

    css="""
.card {
    font-family:
        -apple-system,
        BlinkMacSystemFont,
        "Segoe UI",
        sans-serif;

    font-size: 18px;
    text-align: center;

    color: #222;
    background: #fff;
}

.card-container {
    max-width: 700px;
    margin: 0 auto;
    padding: 18px;
}

.arabic {
    direction: rtl;
    unicode-bidi: isolate;

    font-family:
        "Noto Naskh Arabic",
        "Amiri",
        "Scheherazade New",
        serif;
}

.headword {
    font-size: 44px;
    line-height: 1.5;
    margin-bottom: 4px;
}

.plural {
    font-size: 25px;
    opacity: 0.72;
    margin-bottom: 10px;
}

.definition {
    font-size: 25px;
    font-weight: 600;
    margin: 22px 0;
}

.metadata {
    font-size: 15px;
    line-height: 1.8;
    opacity: 0.75;
}

.label {
    font-weight: 600;
    margin-right: 8px;
}

.root {
    font-size: 19px;
}

.occurrence {
    margin-top: 24px;
    padding-top: 18px;
    border-top: 1px solid #ddd;
}

.occurrence-label {
    font-size: 12px;
    text-transform: uppercase;
    letter-spacing: 0.08em;
    opacity: 0.55;
}

.occurrence-arabic {
    font-size: 31px;
    line-height: 1.9;
    margin: 10px 0;
}

.context-target {
    font-weight: 700;
    color: #b84a3a;
}

.context-english {
    font-size: 15px;
    line-height: 1.65;
    opacity: 0.78;
    margin-top: 10px;
}

.occurrence-gloss {
    font-size: 15px;
    opacity: 0.7;
    margin-top: 8px;
}

.reference {
    font-size: 13px;
    opacity: 0.5;
    margin-top: 6px;
}

.footer {
    margin-top: 24px;
    font-size: 12px;
    opacity: 0.4;
}

#answer {
    margin-top: 18px;
    border: 0;
    border-top: 1px solid #ddd;
}
""",
)

In [ ]:
def anki_text(value):
    if is_blank(value):
        return ""

    return (
        html.escape(
            str(value)
        )
        .replace(
            "\n",
            "<br>"
        )
    )


def anki_context_html(row):
    """
    Render a pause-bounded Arabic context phrase.

    The selected orthographic word is highlighted, while the
    surrounding Tanzil text remains escaped and unchanged.
    """
    before = row.get(
        "Context Before Arabic",
        "",
    )

    target = row.get(
        "Context Target Arabic",
        "",
    )

    after = row.get(
        "Context After Arabic",
        "",
    )

    pieces = []

    if not is_blank(before):
        pieces.append(
            html.escape(
                str(before)
            )
        )

    if not is_blank(target):
        pieces.append(
            '<span class="context-target">'
            + html.escape(
                str(target)
            )
            + "</span>"
        )

    if not is_blank(after):
        pieces.append(
            html.escape(
                str(after)
            )
        )

    if pieces:
        return " ".join(
            pieces
        )

    # Defensive fallback.
    return anki_text(
        row.get(
            "Quranic Form",
            "",
        )
    )


def row_tags(row):
    tags = str(
        row.get(
            "Tags",
            ""
        )
    )

    return [
        tag
        for tag
        in tags.split()
        if tag
    ]

In [ ]:
def download_candidate_audio(
    vocabulary_id,
    record,
    force=True,
):
    audio_path = record.get(
        "Audio Path",
        "",
    )

    if is_blank(audio_path):
        return None

    target_name = audio_media_name(
        vocabulary_id,
        record.get(
            "Example Reference",
            "",
        ),
        audio_path,
    )

    return download_audio(
        audio_path,
        target_name=target_name,
        force=force,
    )


def resolve_audio_for_card(
    vocabulary_id,
    lexicon_row,
    force=True,
):
    """
    Try the occurrence already selected for the card first.

    If its CDN asset is missing, try the remaining ranked
    occurrences for this lexeme. If a fallback is used, return
    its occurrence record too so the displayed Qur'anic word,
    gloss, and reference stay synchronized with the audio.

    Network access therefore happens only for cards that are
    actually being packaged, not for all 4,875 lexemes during
    dataframe construction.
    """

    primary_record = {
        "Quranic Word":
            lexicon_row.get(
                "Quranic Form",
                "",
            ),

        "Occurrence Gloss":
            lexicon_row.get(
                "Occurrence Gloss",
                "",
            ),

        "Example Reference":
            lexicon_row.get(
                "Example Reference",
                "",
            ),

        "Audio Path":
            lexicon_row.get(
                "Audio Path",
                "",
            ),

        "Context Before Arabic":
            lexicon_row.get(
                "Context Before Arabic",
                "",
            ),

        "Context Target Arabic":
            lexicon_row.get(
                "Context Target Arabic",
                "",
            ),

        "Context After Arabic":
            lexicon_row.get(
                "Context After Arabic",
                "",
            ),

        "Context English":
            lexicon_row.get(
                "Context English",
                "",
            ),

        "Waqf Word Count":
            lexicon_row.get(
                "Waqf Word Count",
                "",
            ),
    }

    primary_audio = (
        download_candidate_audio(
            vocabulary_id,
            primary_record,
            force=force,
        )
    )

    if primary_audio is not None:
        return (
            primary_audio,
            primary_record,
            False,
        )

    # The preferred occurrence had no usable audio.
    # Try other ranked occurrences.
    work_row = work.loc[
        vocabulary_id
    ]

    ranked = rank_occurrences(
        vocabulary_id,
        work_row,
    )

    for _, candidate in (
        ranked.iloc[1:].iterrows()
    ):
        record = occurrence_record(
            candidate
        )

        # Skip the same source path if it somehow reappears.
        if (
            record.get("Audio Path")
            == primary_record.get("Audio Path")
        ):
            continue

        audio_file = (
            download_candidate_audio(
                vocabulary_id,
                record,
                force=force,
            )
        )

        if audio_file is not None:
            return (
                audio_file,
                record,
                True,
            )

    return (
        None,
        primary_record,
        False,
    )


def build_anki_package(
    lexicon,
    output_path,
    download_audio=True,
    force_audio=True,
):
    deck = genanki.Deck(
        DECK_ID,
        DECK_NAME,
    )

    media_files = []

    lexicon = (
        lexicon
        .sort_values(
            "Frequency Rank"
        )
    )

    for number, (
        vocabulary_id,
        row,
    ) in enumerate(
        lexicon.iterrows(),
        start=1,
    ):

        # Work on a copy so a fallback audio occurrence does not
        # mutate final_lexicon.
        card_row = row.copy()

        audio_field = ""

        if (
            download_audio
            and not is_blank(
                card_row["Audio Path"]
            )
        ):
            (
                audio_file,
                audio_record,
                used_fallback,
            ) = resolve_audio_for_card(
                vocabulary_id,
                card_row,
                force=force_audio,
            )

            if audio_file is not None:
                media_files.append(
                    str(audio_file)
                )

                audio_field = (
                    "[sound:"
                    + audio_file.name
                    + "]"
                )

                # Keep text/reference synchronized with the
                # recording if a fallback occurrence was needed.
                if used_fallback:
                    card_row[
                        "Quranic Form"
                    ] = audio_record.get(
                        "Quranic Word",
                        card_row[
                            "Quranic Form"
                        ],
                    )

                    card_row[
                        "Occurrence Gloss"
                    ] = audio_record.get(
                        "Occurrence Gloss",
                        card_row[
                            "Occurrence Gloss"
                        ],
                    )

                    card_row[
                        "Example Reference"
                    ] = audio_record.get(
                        "Example Reference",
                        card_row[
                            "Example Reference"
                        ],
                    )

                    card_row[
                        "Audio Path"
                    ] = audio_record.get(
                        "Audio Path",
                        card_row[
                            "Audio Path"
                        ],
                    )

                    for context_column in [
                        "Context Before Arabic",
                        "Context Target Arabic",
                        "Context After Arabic",
                        "Context English",
                        "Waqf Word Count",
                    ]:
                        card_row[
                            context_column
                        ] = audio_record.get(
                            context_column,
                            card_row.get(
                                context_column,
                                "",
                            ),
                        )

        fields = [
            anki_text(
                vocabulary_id
            ),
            anki_text(
                card_row["Headword"]
            ),
            anki_text(
                card_row["Plural"]
            ),
            anki_text(
                card_row["Definition"]
            ),
            anki_text(
                card_row["Root"]
            ),
            anki_text(
                card_row["POS"]
            ),
            str(
                int(
                    card_row["Frequency"]
                )
            ),
            str(
                int(
                    card_row[
                        "Frequency Rank"
                    ]
                )
            ),
            anki_context_html(
                card_row
            ),
            anki_text(
                card_row.get(
                    "Context English",
                    card_row[
                        "Occurrence Gloss"
                    ],
                )
            ),
            anki_text(
                card_row[
                    "Example Reference"
                ]
            ),
            audio_field,
            anki_text(
                card_row[
                    "Observed Forms"
                ]
            ),
            anki_text(
                card_row[
                    "Headword Source"
                ]
            ),
            anki_text(
                card_row[
                    "Definition Source"
                ]
            ),
            str(
                bool(
                    card_row[
                        "Auto QA Flag"
                    ]
                )
            ),
        ]

        note = genanki.Note(
            model=anki_model,

            fields=fields,

            tags=row_tags(
                card_row
            ),

            # Stable identity across future regenerations.
            guid=genanki.guid_for(
                "CompleteQuranVocabulary",
                str(
                    vocabulary_id
                ),
            ),
        )

        deck.add_note(
            note
        )

        if (
            number % 25
            == 0
        ):
            print(
                f"Processed "
                f"{number}/"
                f"{len(lexicon)}"
            )

    package = genanki.Package(
        deck
    )

    package.media_files = (
        sorted(
            set(
                media_files
            )
        )
    )

    package.write_to_file(
        str(
            output_path
        )
    )

    print(
        "Created:",
        output_path.resolve()
    )


In [ ]:
TEST_CARD_COUNT = 100

test_lexicon = (
    final_lexicon
    .sort_values(
        "Frequency Rank"
    )
    .head(
        TEST_CARD_COUNT
    )
)

build_anki_package(
    test_lexicon,
    TEST_APKG_PATH,
    download_audio=True,
    force_audio=True,
)

In [ ]:
def inspect_card(
    headword,
    max_rows=20,
):
    matches = work[
        work["Headword"]
        == headword
    ]

    if matches.empty:
        print(
            "Headword not found:",
            headword,
        )
        return

    for vocabulary_id, row in (
        matches.iterrows()
    ):
        print(
            "\n" + "=" * 72
        )

        print(
            "Vocabulary ID:",
            vocabulary_id,
        )

        print(
            "Headword:",
            row["Headword"],
        )

        print(
            "Selected lexical segment:",
            row.get(
                "Lexical Occurrence",
                "",
            ),
        )

        print(
            "Selected complete word:",
            row.get(
                "Quranic Word",
                "",
            ),
        )

        print(
            "Selected reference:",
            row.get(
                "Example Reference",
                "",
            ),
        )

        print(
            "Selected audio:",
            row.get(
                "Audio Path",
                "",
            ),
        )


        print(
            "Selected waqf length:",
            row.get(
                "Waqf Word Count",
                "",
            ),
        )

        print(
            "Selected context:",
            " ".join(
                str(part)
                for part in [
                    row.get(
                        "Context Before Arabic",
                        "",
                    ),
                    row.get(
                        "Context Target Arabic",
                        "",
                    ),
                    row.get(
                        "Context After Arabic",
                        "",
                    ),
                ]
                if str(part).strip()
            ),
        )

        print(
            "Context English:",
            row.get(
                "Context English",
                "",
            ),
        )

        ranked = rank_occurrences(
            vocabulary_id,
            row,
        )

        columns = [
            "Surah",
            "Ayah",
            "Word",
            "Segment",
            "Arabic",
            "Quranic Word",
            "English",
            "Cleaned English",
            "QACSegmentCount",
            "VocabularySegmentCount",
            "PrefixCount",
            "SuffixCount",
            "Number",
            "HasPronounSuffix",
            "_vocalized_similarity",
            "Waqf Word Count",
            "Context Target Arabic",
            "Audio",
        ]

        existing_columns = [
            column
            for column in columns
            if column
            in ranked.columns
        ]

        display(
            ranked[
                existing_columns
            ].head(
                max_rows
            )
        )


# Useful problem cases while we validate occurrence/audio selection:
# inspect_card("فِي")
# inspect_card("كَانَ")
# inspect_card("أَرْض")
# inspect_card("ٱلَّذِي")
# inspect_card("آمَنَ")
# inspect_card("قَد")
